# CIC-IDS-2017 Feature Extraction & Labeling Pipeline
This notebook processes raw PCAP files from the CIC-IDS-2017 dataset (Tuesday and Wednesday).
It utilizes a hybrid extraction pipeline (NFStream + Zeek) to extract encrypted traffic metadata and universal features.
Finally, it applies the corrected 5-tuple labels provided by DistriNet to fix the original CICFlowMeter flaws.

# Phase 1: Environment Initialization
- mounts Google Drive
- installs Zeek
- clones our hybrid feature extraction pipeline.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')


In [ ]:

# 1. Define workspace path
WORKSPACE_DIR = '/content/drive/MyDrive/CIC-IDS-2017_Project'
print(f"[*] Workspace initialization target: {WORKSPACE_DIR}")

# 2. CREATE the directories FIRST (Fixes the [Errno 2] issue)
os.makedirs(WORKSPACE_DIR, exist_ok=True)

# 3. Now it is safe to change directory into the workspace
%cd {WORKSPACE_DIR}

# 4. Clone the pipeline repository into the Drive workspace
if not os.path.exists("encrypted-traffic-analytics"):
    !git clone https://github.com/hedietahmouresi/encrypted-traffic-analytics.git
else:
    print("[*] Repository already cloned.")

%cd encrypted-traffic-analytics

os.makedirs(f'{WORKSPACE_DIR}/encrypted-traffic-analytics/data/raw', exist_ok=True)
os.makedirs(f'{WORKSPACE_DIR}/encrypted-traffic-analytics/data/processed', exist_ok=True)
os.makedirs(f'{WORKSPACE_DIR}/encrypted-traffic-analytics/data/labels', exist_ok=True)

# 5. Install Python dependencies
!echo "[*] Installing Python dependencies..."
!pip install -r requirements.txt > /dev/null
print("[+] Phase 1 Complete and verified!")

In [ ]:
%cd encrypted-traffic-analytics

# 1. Install Zeek (Colab runs Ubuntu 22.04)
!echo "[*] Installing Zeek (Non-Interactive)..."
!curl -fsSL https://download.opensuse.org/repositories/security:zeek/xUbuntu_22.04/Release.key | gpg --dearmor | sudo tee /etc/apt/trusted.gpg.d/security_zeek.gpg > /dev/null
!echo 'deb http://download.opensuse.org/repositories/security:/zeek/xUbuntu_22.04/ /' | sudo tee /etc/apt/sources.list.d/security:zeek.list
!sudo apt-get update -qq
# This environment variable forces apt-get to skip any interactive menus
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -yq zeek > /dev/null

# Add Zeek to the system PATH so your script can run it
os.environ['PATH'] += ':/opt/zeek/bin'

!echo "[+] Phase 1 Complete!"

# Phase 2: Downloading PCAPs from Hugging Face
Now we will pull the massive Tuesday and Wednesday PCAPs, along with the original CSV files, directly into your Google Drive. Using Hugging Face is incredibly efficient for this.

In [ ]:
import os

WORKSPACE_DIR = '/content/drive/MyDrive/CIC-IDS-2017_Project'
%cd {WORKSPACE_DIR}/encrypted-traffic-analytics

print("[*] Starting downloads via wget (Resumable)...")

print("\n[*] Downloading Tuesday PCAP...")
!wget -c https://huggingface.co/datasets/Ariasyah/cic-ids-2017/resolve/main/pcap/Tuesday-WorkingHours.pcap -P ./data/raw/

print("\n[*] Downloading Wednesday PCAP...")
!wget -c https://huggingface.co/datasets/Ariasyah/cic-ids-2017/resolve/main/pcap/Wednesday-workingHours.pcap -P ./data/raw/


# Phase 3 : Feature Extraction
- run your HybridExtractor
- map the original CIC-IDS-2017 labels

In [ ]:
def apply_heuristic_labels(df, day, output_path):
    print(f"[*] Applying Schedule-Based Heuristic Labels for {day}...")

    df['datetime'] = pd.to_datetime(df['bidirectional_first_seen_ms'], unit='ms', utc=True)
    df['datetime'] = df['datetime'].dt.tz_convert('America/Halifax')

    df['label'] = 'BENIGN'

    if day.lower() == "tuesday":
        # Tuesday Attacks: Brute Force
        # FTP-Patator
        mask_ftp = (
            (df['datetime'] >= '2017-07-04 09:20:00') & (df['datetime'] <= '2017-07-04 10:20:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.50') & (df['dst_port'] == 21)
        )
        df.loc[mask_ftp, 'label'] = 'FTP-Patator'

        # SSH-Patator
        mask_ssh = (
            (df['datetime'] >= '2017-07-04 14:00:00') & (df['datetime'] <= '2017-07-04 15:00:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.50') & (df['dst_port'] == 22)
        )
        df.loc[mask_ssh, 'label'] = 'SSH-Patator'

    elif day.lower() == "wednesday":
        # Wednesday Attacks: DoS / DDoS / Web
        mask_slowloris = (
            (df['datetime'] >= '2017-07-05 09:47:00') & (df['datetime'] <= '2017-07-05 10:10:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.50') & (df['dst_port'] == 80)
        )
        df.loc[mask_slowloris, 'label'] = 'DoS-Slowloris'

        mask_slowhttp = (
            (df['datetime'] >= '2017-07-05 10:14:00') & (df['datetime'] <= '2017-07-05 10:35:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.50') & (df['dst_port'] == 80)
        )
        df.loc[mask_slowhttp, 'label'] = 'DoS-SlowHTTPTest'

        mask_hulk = (
            (df['datetime'] >= '2017-07-05 10:43:00') & (df['datetime'] <= '2017-07-05 11:00:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.50') & (df['dst_port'] == 80)
        )
        df.loc[mask_hulk, 'label'] = 'DoS-Hulk'

        mask_goldeneye = (
            (df['datetime'] >= '2017-07-05 11:10:00') & (df['datetime'] <= '2017-07-05 11:23:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.50') & (df['dst_port'] == 80)
        )
        df.loc[mask_goldeneye, 'label'] = 'DoS-GoldenEye'

        mask_heartbleed = (
            (df['datetime'] >= '2017-07-05 15:12:00') & (df['datetime'] <= '2017-07-05 15:32:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.51') & (df['dst_port'] == 444)
        )
        df.loc[mask_heartbleed, 'label'] = 'Heartbleed'

        mask_ddos = (
            (df['datetime'] >= '2017-07-05 15:56:00') & (df['datetime'] <= '2017-07-05 16:25:00') &
            (df['src_ip'] == '172.16.0.1') & (df['dst_ip'] == '192.168.10.50') & (df['dst_port'] == 80)
        )
        df.loc[mask_ddos, 'label'] = 'DDoS'

    else:
        print(f"[!] Warning: No schedule defined for {day}. All traffic marked BENIGN.")

    df['datetime'] = df['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')

    print("[*] Stripping contaminated routing, temporal, and topological data...")
    
    contaminants = [
        # Identifiers & Topo
        'id', 'expiration_id', 'uid',
        'src_ip', 'dst_ip', 'src_mac', 'dst_mac', 
        'src_oui', 'dst_oui', 'vlan_id', 'tunnel_id',
        
        # Ports
        'src_port', 'dst_port',
        
        # Timestamps
        'datetime', 'ts', 'ts_ms',
        'bidirectional_first_seen_ms', 'bidirectional_last_seen_ms',
        'src2dst_first_seen_ms', 'src2dst_last_seen_ms',
        'dst2src_first_seen_ms', 'dst2src_last_seen_ms',
        
        # Highly Specific Metadata
        'client_fingerprint', 'server_fingerprint',
        'requested_server_name', 'user_agent'
    ]
    
    cols_to_drop = [col for col in contaminants if col in df.columns]
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"[*] Labeling complete. Distribution:\n{df['label'].value_counts()}")

    df.to_csv(output_path, index=False)
    return df

In [ ]:
# 1. Ensure we are in the correct workspace
WORKSPACE_DIR = '/content/drive/MyDrive/CIC-IDS-2017_Project'
%cd {WORKSPACE_DIR}/encrypted-traffic-analytics

import os
import pandas as pd
import gc
from src.feature_extractor import HybridExtractor

# 2. Define the exact file paths for the downloaded data
days = ["Tuesday", "Wednesday"]
raw_pcaps = {
    "Tuesday": "./data/raw/Tuesday-WorkingHours.pcap",
    "Wednesday": "./data/raw/Wednesday-workingHours.pcap"
}
label_files = {
    "Tuesday": "./data/labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet",
    "Wednesday": "./data/labels/Wednesday-workingHours.pcap_ISCX.csv.parquet"
}

# 2. Execute the pipeline for each day
for day in days:
    pcap_path = raw_pcaps[day]
    label_path = label_files[day]
    out_dir = f"./data/processed/{day}"
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*50}")
    print(f"[*] Starting Phase 3 Pipeline for {day}")
    print(f"{'='*50}")

    if os.path.exists(pcap_path) and os.path.exists(label_path):
        extractor = HybridExtractor(pcap_path=pcap_path, output_dir=out_dir)
        unified_df = extractor.process()
        apply_heuristic_labels(unified_df, day, final_out_csv)

        del unified_df
        import gc
        gc.collect()
    else:
        print(f"[!] Missing PCAP or Label file for {day}. Check the paths.")

print("\n[🚀] ALL PHASES COMPLETE! Your ready-to-train datasets are in /data/processed/")